In [4]:
#import Libraries

import zipfile
from io import BytesIO
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import ast
from torch_geometric.data import HeteroData

In [3]:
fname_base = '/mnt/nas/pisani/dataset_mumin/mumin-small.v0.zip' 

def read_pickle_from_zip_folder(fname):

    # Open the zip file
    with zipfile.ZipFile(fname_base, 'r') as zip_ref:
        
        with zip_ref.open(fname) as file:
            # Read the file content into a pandas dataframe
            return pd.read_pickle(BytesIO(file.read()))
        


            

In [5]:
#Read the data

# nodes
df_C = read_pickle_from_zip_folder('claim.pickle')
df_T = read_pickle_from_zip_folder('tweet.pickle')
df_R = read_pickle_from_zip_folder('reply.pickle')
df_U = read_pickle_from_zip_folder('user.pickle')
df_H = read_pickle_from_zip_folder('hashtag.pickle')
df_A = read_pickle_from_zip_folder('article.pickle')
df_I = read_pickle_from_zip_folder('image.pickle')

# edges
df_TC = read_pickle_from_zip_folder('tweet_discusses_claim.pickle')
df_TR_r = read_pickle_from_zip_folder('reply_reply_to_tweet.pickle')
df_TR_q = read_pickle_from_zip_folder('reply_quote_of_tweet.pickle')
df_TH = read_pickle_from_zip_folder('tweet_has_hashtag_hashtag.pickle')
df_TA = read_pickle_from_zip_folder('tweet_has_article_article.pickle')
df_TI = read_pickle_from_zip_folder('tweet_has_image_image.pickle')
df_TU_m = read_pickle_from_zip_folder('tweet_mentions_user.pickle')
df_UT = read_pickle_from_zip_folder('user_posted_tweet.pickle')
df_UR = read_pickle_from_zip_folder('user_posted_reply.pickle')
df_UT_r = read_pickle_from_zip_folder('user_retweeted_tweet.pickle')
df_UU_f = read_pickle_from_zip_folder('user_follows_user.pickle')
df_UU_m = read_pickle_from_zip_folder('user_mentions_user.pickle')
df_UH = read_pickle_from_zip_folder('user_has_hashtag_hashtag.pickle')

In [6]:
#Label Encoder for ground truth
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn

def label_encoding(df, col):
    label_encoder = LabelEncoder()
    encoded_data = label_encoder.fit_transform(df[col])
    # original_data = label_encoder.inverse_transform(encoded_data)
    #print(original_data)
    return encoded_data

def learnable_embedding(df, emb_dim):
    emb = nn.Embedding(df.shape[0], emb_dim)
    return emb

def edges_encoding(df):
    return torch.tensor(df.values.T)

def edges_rev_encoding(df):
    df_rev = df[['tgt', 'src']]
    return torch.tensor(df_rev.values.T)



Data_1 --> only edges in the graph (no mps and no node features)

In [5]:
# NODES

#Adding Learnable Embeddings (instead of identity matrix)

emb_dim = 128

claim_emb = learnable_embedding(df_C, emb_dim)
tweet_emb = learnable_embedding(df_T, emb_dim)
retweet_emb = learnable_embedding(df_R, emb_dim)
user_emb = learnable_embedding(df_U, emb_dim)
hashtag_emb = learnable_embedding(df_H, emb_dim)
article_emb = learnable_embedding(df_A, emb_dim)
image_emb = learnable_embedding(df_I, emb_dim)


# EDGES

# tweet_discusses_claim
TC_tensor = edges_encoding(df_TC)
CT_tensor = edges_rev_encoding(df_TC)
torch.save(TC_tensor, '../data/edgelists/TC_tensor.pt')
torch.save(CT_tensor, '../data/edgelists/CT_tensor.pt')

# reply_reply_to_tweet
TR_r_tensor = edges_encoding(df_TR_r)
RT_r_tensor = edges_rev_encoding(df_TR_r)
torch.save(TR_r_tensor, '../data/edgelists/TR_r_tensor.pt')
torch.save(RT_r_tensor, '../data/edgelists/RT_r_tensor.pt')

# reply_quote_of_tweet
TR_q_tensor = edges_encoding(df_TR_q)
RT_q_tensor = edges_rev_encoding(df_TR_q)
torch.save(TR_q_tensor, '../data/edgelists/TR_q_tensor.pt')
torch.save(RT_q_tensor, '../data/edgelists/RT_q_tensor.pt')

# tweet_has_hashtag_hashtag
TH_tensor = edges_encoding(df_TH)
HT_tensor = edges_rev_encoding(df_TH)
torch.save(TH_tensor, '../data/edgelists/TH_tensor.pt')
torch.save(HT_tensor, '../data/edgelists/HT_tensor.pt')

# tweet_has_article_article
TA_tensor = edges_encoding(df_TA)
AT_tensor = edges_rev_encoding(df_TA)
torch.save(TA_tensor, '../data/edgelists/TA_tensor.pt')
torch.save(AT_tensor, '../data/edgelists/AT_tensor.pt')

# tweet_has_image_image
TI_tensor = edges_encoding(df_TI)
IT_tensor = edges_rev_encoding(df_TI)
torch.save(TI_tensor, '../data/edgelists/TI_tensor.pt')
torch.save(IT_tensor, '../data/edgelists/IT_tensor.pt')

# tweet_mentions_user
TU_m_tensor = edges_encoding(df_TU_m)
UT_m_tensor = edges_rev_encoding(df_TU_m)
torch.save(TU_m_tensor, '../data/edgelists/TU_m_tensor.pt')
torch.save(UT_m_tensor, '../data/edgelists/UT_m_tensor.pt')

# user_posted_tweet
UT_tensor = edges_encoding(df_UT)
TU_tensor = edges_rev_encoding(df_UT)
torch.save(UT_tensor, '../data/edgelists/UT_tensor.pt')
torch.save(TU_tensor, '../data/edgelists/TU_tensor.pt')

# user_posted_reply
UR_tensor = edges_encoding(df_UR)
RU_tensor = edges_rev_encoding(df_UR)
torch.save(UR_tensor, '../data/edgelists/UR_tensor.pt')
torch.save(RU_tensor, '../data/edgelists/RU_tensor.pt')

# user_retweeted_tweet
UT_r_tensor = edges_encoding(df_UT_r)
TU_r_tensor = edges_rev_encoding(df_UT_r)
torch.save(UT_r_tensor, '../data/edgelists/UT_r_tensor.pt')
torch.save(TU_r_tensor, '../data/edgelists/TU_r_tensor.pt')

# user_follows_user
UU_f_tensor = edges_encoding(df_UU_f)
UU_f_rev_tensor = edges_rev_encoding(df_UU_f)
torch.save(UU_f_tensor, '../data/edgelists/UU_f_tensor.pt')
torch.save(UU_f_rev_tensor, '../data/edgelists/UU_f_rev_tensor.pt')

# user_mentions_user
UU_m_tensor = edges_encoding(df_UU_m)
UU_m_rev_tensor = edges_rev_encoding(df_UU_m)
torch.save(UU_m_tensor, '../data/edgelists/UU_m_tensor.pt')
torch.save(UU_m_rev_tensor, '../data/edgelists/UU_m_rev_tensor.pt')

# user_has_hashtag_hashtag
UH_tensor = edges_encoding(df_UH)
HU_tensor = edges_rev_encoding(df_UH)
torch.save(UH_tensor, '../data/edgelists/UH_tensor.pt')
torch.save(HU_tensor, '../data/edgelists/HU_tensor.pt')




In [12]:
# Initialize HeteroData object
data = HeteroData()

# NODES

data['claim'].x = claim_emb.weight
data['claim'].y = label_encoding(df_C, 'label')
data['tweet'].x = tweet_emb.weight
data['retweet'].x = retweet_emb.weight
data['user'].x = user_emb.weight
data['hashtag'].x = hashtag_emb.weight
data['article'].x = article_emb.weight
data['image'].x = image_emb.weight

# EDGES

data['tweet', 'discusses', 'claim'].edge_index = TC_tensor
data['claim', 'is_discussed_by', 'tweet'].edge_index = CT_tensor

data['reply', 'reply_to', 'tweet'].edge_index = TR_r_tensor
data['tweet', 'is_replied_by', 'reply'].edge_index = RT_r_tensor

data['reply', 'quote_of', 'tweet'].edge_index = TR_q_tensor
data['tweet', 'is_quoted_by', 'reply'].edge_index = RT_q_tensor

data['tweet', 'has_hashtag', 'hashtag'].edge_index = TH_tensor
data['hashtag', 'is_hashtag_of', 'tweet'].edge_index = HT_tensor

data['tweet', 'has_article', 'article'].edge_index = TA_tensor
data['article', 'is_article_of', 'tweet'].edge_index = AT_tensor

data['tweet', 'has_image', 'image'].edge_index = TI_tensor
data['image', 'is_image_of', 'tweet'].edge_index = IT_tensor

data['tweet', 'mentions', 'user'].edge_index = TU_m_tensor
data['user', 'is_mentioned_by', 'tweet'].edge_index = UT_m_tensor

data['user', 'posted', 'tweet'].edge_index = UT_tensor
data['tweet', 'is_posted_by', 'user'].edge_index = TU_tensor

data['user', 'posted', 'reply'].edge_index = UR_tensor
data['reply', 'is_posted_by', 'user'].edge_index = RU_tensor

data['user', 'retweeted', 'tweet'].edge_index = UT_r_tensor
data['tweet', 'is_retweeted_by', 'user'].edge_index = TU_r_tensor

data['user', 'follows', 'user'].edge_index = UU_f_tensor
data['user', 'is_followed_by', 'user'].edge_index = UU_f_rev_tensor

data['user', 'mentions', 'user'].edge_index = UU_m_tensor
data['user', 'is_mentioned_by', 'user'].edge_index = UU_m_rev_tensor

data['user', 'has_hashtag', 'hashtag'].edge_index = UH_tensor
data['hashtag', 'is_hashtag_of', 'user'].edge_index = HU_tensor



In [7]:
data

HeteroData(
  claim={
    x=[2168, 128],
    y=[2168],
  },
  tweet={ x=[4340, 128] },
  retweet={ x=[195459, 128] },
  user={ x=[153168, 128] },
  hashtag={ x=[28091, 128] },
  article={ x=[1457, 128] },
  image={ x=[1020, 128] },
  (tweet, discusses, claim)={ edge_index=[2, 5081] },
  (claim, is_discussed_by, tweet)={ edge_index=[2, 5081] },
  (reply, reply_to, tweet)={ edge_index=[2, 90196] },
  (tweet, is_replied_by, reply)={ edge_index=[2, 90196] },
  (reply, quote_of, tweet)={ edge_index=[2, 101216] },
  (tweet, is_quoted_by, reply)={ edge_index=[2, 101216] },
  (tweet, has_hashtag, hashtag)={ edge_index=[2, 2289] },
  (hashtag, is_hashtag_of, tweet)={ edge_index=[2, 2289] },
  (tweet, has_article, article)={ edge_index=[2, 1898] },
  (article, is_article_of, tweet)={ edge_index=[2, 1898] },
  (tweet, has_image, image)={ edge_index=[2, 1028] },
  (image, is_image_of, tweet)={ edge_index=[2, 1028] },
  (tweet, mentions, user)={ edge_index=[2, 1119] },
  (user, is_mentioned_by, twe

Data_2 --> only edges in the graph + meta-paths

In [14]:
from torch_geometric.transforms import AddMetaPaths

#Add metapaths # CTUTC, CTHTC

metapaths = [[('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_posted_by', 'user'), 
              ('user', 'posted', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTUTC
             [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'has_hashtag', 'hashtag'), 
              ('hashtag', 'is_hashtag_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')]] #CTHTC


data = AddMetaPaths(metapaths, weighted=True)(data)

In [15]:
data

HeteroData(
  metapath_dict={
    (claim, metapath_0, claim)=[4],
    (claim, metapath_1, claim)=[4],
  },
  claim={
    x=[2168, 128],
    y=[2168],
  },
  tweet={ x=[4340, 128] },
  retweet={ x=[195459, 128] },
  user={ x=[153168, 128] },
  hashtag={ x=[28091, 128] },
  article={ x=[1457, 128] },
  image={ x=[1020, 128] },
  (tweet, discusses, claim)={ edge_index=[2, 5081] },
  (claim, is_discussed_by, tweet)={ edge_index=[2, 5081] },
  (reply, reply_to, tweet)={ edge_index=[2, 90196] },
  (tweet, is_replied_by, reply)={ edge_index=[2, 90196] },
  (reply, quote_of, tweet)={ edge_index=[2, 101216] },
  (tweet, is_quoted_by, reply)={ edge_index=[2, 101216] },
  (tweet, has_hashtag, hashtag)={ edge_index=[2, 2289] },
  (hashtag, is_hashtag_of, tweet)={ edge_index=[2, 2289] },
  (tweet, has_article, article)={ edge_index=[2, 1898] },
  (article, is_article_of, tweet)={ edge_index=[2, 1898] },
  (tweet, has_image, image)={ edge_index=[2, 1028] },
  (image, is_image_of, tweet)={ edge_index

Data_3 --> Heterodata object with nodes/edges (no meta-paths) + node attributes

In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# preprocessing of attributes

# scaling of numeric columns
def scale_numeric(df, col):
    scaler = MinMaxScaler()
    df[col+'_scaled'] = scaler.fit_transform(df[[col]])
    df.drop([col], axis=1, inplace=True)
    return df


# conversion from datetime64[ns] to timestamp + normalization
# boost: replace nan values with the mean of the column instead of epoch start to avoid otliers (MinMax Scaling is sensitive to outliers)
def convert_datetime_to_timestamp(df, col):
    #df[col+'_timestamp'] = df[col].apply(lambda x: x.timestamp() if pd.notnull(x) else pd.Timestamp('1970-01-01 00:00:00').timestamp()) 
    mean_date = df[col].mean()
    df[col] = df[col].fillna(mean_date)
    df[col+'_timestamp'] = df[col].apply(lambda x: x.timestamp()) 
    scaler = MinMaxScaler()
    df[col+'_timestamp_scaled'] = scaler.fit_transform(df[[col+'_timestamp']])
    df.drop([col, col+'_timestamp'], axis=1, inplace=True)
    return df


# encoding of short texts
# boost: apply PCA after SBERT on the individual attribute. Apply the mean to nan values
def encoding_short_text(df, col, target_dim=28):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    #df[col+'_encoded'] = df.apply(lambda x: model.encode(x[col]), axis=1)

     # Convert the column to string if it's categorical
    if isinstance(df[col].dtype, pd.CategoricalDtype):
    #if pd.api.types.is_categorical_dtype(df[col]):
        df[col] = df[col].astype(str)

    # Generate embeddings for text entries
    def get_embedding(text_or_list):
        ''' if pd.isna(text_or_list) or text_or_list is None:
            return None '''
        if isinstance(text_or_list, str):
            if text_or_list == "":
                return None
            return model.encode(text_or_list)
        if isinstance(text_or_list, list):
            embeddings = [model.encode(text) for text in text_or_list if text != ""]
            if embeddings:
                return np.mean(embeddings, axis=0)
            return None
        return None
    
    embeddings = df[col].apply(get_embedding)   
    #non_nan_embeddings = embeddings.dropna().values
    non_nan_embeddings = [e for e in embeddings if e is not None]

    if len(non_nan_embeddings) > 0:
        embeddings_matrix = np.vstack(non_nan_embeddings)
        mean_embedding = np.mean(embeddings_matrix, axis=0)

        def replace_missing(embedding):
            if embedding is None:
                return mean_embedding
            else:
                return embedding

        embeddings = embeddings.apply(replace_missing)

    embeddings_matrix = np.vstack(embeddings)
    pca = PCA(n_components=target_dim)
    reduced_embeddings = pca.fit_transform(embeddings_matrix)
    df[col + '_encoded'] = [embedding.tolist() for embedding in reduced_embeddings] #Store the reduced embeddings in a single column as lists
    df.drop([col], axis=1, inplace=True)
    return df



def parsing_embedding_float(df, col):
    df[col+'_parsed'] = df.apply(lambda x: torch.tensor(x[col], dtype=torch.float32), axis=1)
    df.drop([col], axis=1, inplace=True)
    return df

def parsing_embedding_int(df, col):
    df[col+'_parsed'] = df.apply(lambda x: torch.tensor(x[col], dtype=torch.int32), axis=1)
    df.drop([col], axis=1, inplace=True)
    return df


def one_hot_encoding(df, col):
    df[col] = df[col].astype(str) #added for int values (cluster_labels)
    if df[col].dtype.name == 'category':
        unique_categories = df[col].cat.categories.tolist()
    else:
        unique_categories = df[col].unique().tolist()
    num_categories = len(unique_categories)
    category_to_index = {category: idx for idx, category in enumerate(unique_categories)}
    
    def one_hot_encode(category):
        category = str(category)  # Ensure the category is treated as a string
        one_hot_vector = np.zeros(num_categories, dtype=int)
        one_hot_vector[category_to_index[category]] = 1
        #return one_hot_vector
        return torch.tensor(one_hot_vector, dtype=torch.int32)
    
    df[col+'_ohe'] = df.apply(lambda x: one_hot_encode(x[col]), axis=1)
    df.drop([col], axis=1, inplace=True)
    return df
  

/home/martirano/.venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [8]:
def encoding_attributes(df):
    tensors = []
    for col in df.columns:
        print('##### Processing column ', col, ' #####')
        if df[col].dtype == 'int64':
            tensors.append(torch.tensor(df[col].values, dtype=torch.int32).unsqueeze(1))
        elif df[col].dtype == 'float64':
            tensors.append(torch.tensor(df[col].values, dtype=torch.float32).unsqueeze(1))
        elif df[col].dtype == 'bool':
            tensors.append(torch.tensor(df[col].values, dtype=torch.bool).unsqueeze(1))
        elif df[col].dtype == 'object':
            embedding_tensors = df[col].apply(lambda x: torch.tensor(x, dtype=torch.float32) if not isinstance(x, torch.Tensor) else x)
            embedding_stack = torch.stack(embedding_tensors.tolist())  # Convert to list before stacking
            tensors.append(embedding_stack)
        elif df[col].dtype == 'category':
            enc = LabelEncoder()
            encoded_values = enc.fit_transform(df[col])
            tensors.append(torch.tensor(encoded_values, dtype=torch.int32).unsqueeze(1))
        else:
            raise ValueError(f"Unsupported column type: {df[col].dtype}")

    # Concatenate all tensors along the last dimension
    return torch.cat(tensors, dim=1)

In [43]:
# Preprocessing node features

# claim
print('##### preprocessing claim fetaures... #####')
df_C_sub = df_C.drop(columns=['label', 'reviewer_emb', 'train_mask', 'val_mask', 'test_mask'], axis=1)
df_C_sub = parsing_embedding_float(df_C_sub, 'embedding')
#df_C_sub = parsing_embedding_int(df_C_sub, 'reviewer_emb')
df_C_sub = encoding_short_text(df_C_sub, 'reviewers') #114 distinct values
df_C_sub = convert_datetime_to_timestamp(df_C_sub, 'date')
df_C_sub = one_hot_encoding(df_C_sub,'language') #35 distinct values
df_C_sub = encoding_short_text(df_C_sub, 'keywords')
df_C_sub = encoding_short_text(df_C_sub, 'cluster_keywords', target_dim=64) #26 distinct values, but multiple strings each
#df_C_sub.to_csv('../data/features/CX.csv', index=False)
CX = encoding_attributes(df_C_sub)
torch.save(CX, '../data/features/CX_tensor.pt')
print('claim:', CX.shape)
print(torch.max(CX))

print('##### preprocessing claim labels... #####')
C_labels = label_encoding(df_C, 'label').tolist()
CY = torch.tensor(C_labels)
torch.save(CY, '../data/features/CY_tensor.pt')
print('claim labels:', CY.shape)


# tweet
print('##### preprocessing tweet fetaures... #####')
df_T_sub = df_T.drop(columns=['tweet_id', 'text', 'lang'], axis=1)
df_T_sub = parsing_embedding_float(df_T_sub, 'text_emb')
df_T_sub['lang_emb'] = df_T_sub.apply(lambda x: torch.tensor(x['lang_emb'], dtype=torch.int32), axis=1)
df_T_sub = convert_datetime_to_timestamp(df_T_sub, 'created_at')
df_T_sub = encoding_short_text(df_T_sub, 'source') #136 distinct values
df_T_sub = scale_numeric(df_T_sub, 'num_retweets')
df_T_sub = scale_numeric(df_T_sub, 'num_replies')
df_T_sub = scale_numeric(df_T_sub, 'num_quote_tweets')
#df_T_sub.to_csv('../data/features/TX.csv', index=False)
TX = encoding_attributes(df_T_sub)
torch.save(TX, '../data/features/TX_tensor.pt')
print('tweet:', TX.shape)
print()


# reply
print('##### preprocessing reply fetaures... #####')
df_R_sub = df_R.drop(columns=['tweet_id', 'text', 'lang','source'], axis=1)
df_R_sub = parsing_embedding_float(df_R_sub, 'text_emb')
df_R_sub['lang_emb'] = df_R_sub.apply(lambda x: torch.tensor(x['lang_emb'], dtype=torch.int32), axis=1)
df_R_sub = convert_datetime_to_timestamp(df_R_sub, 'created_at')
df_R_sub = scale_numeric(df_R_sub, 'num_retweets')
df_R_sub = scale_numeric(df_R_sub, 'num_replies')
df_R_sub = scale_numeric(df_R_sub, 'num_quote_tweets')
#df_R_sub.to_csv('../data/features/RX.csv', index=False)
RX = encoding_attributes(df_R_sub)
torch.save(RX, '../data/features/RX_tensor.pt')
print('reply:', RX.shape)
print()


# user
print('##### preprocessing user fetaures... #####')
df_U_sub = df_U.drop(columns=['user_id', 'username', 'description', 'url',	'name'], axis=1)
df_U_sub = convert_datetime_to_timestamp(df_U_sub, 'created_at')
df_U_sub = scale_numeric(df_U_sub, 'num_followers')
df_U_sub = scale_numeric(df_U_sub, 'num_followees')
df_U_sub = scale_numeric(df_U_sub, 'num_tweets')
df_U_sub = scale_numeric(df_U_sub, 'num_listed')
df_U_sub = encoding_short_text(df_U_sub, 'location') 
df_U_sub['description_emb'] = df_U_sub.apply(lambda x: torch.tensor(x['description_emb'], dtype=torch.float32), axis=1)
#df_U_sub.to_csv('../data/features/UX.csv', index=False)
UX = encoding_attributes(df_U_sub)
torch.save(UX, '../data/features/UX_tensor.pt')
print('user:', UX.shape)
print()


# hashtag
print('##### preprocessing hashtag fetaures... #####')
df_H_sub = encoding_short_text(df_H, 'tag') 
#df_H_sub.to_csv('../data/features/HX.csv', index=False)
HX = encoding_attributes(df_H_sub)
torch.save(HX, '../data/features/HX_tensor.pt')
print('hashtag:', HX.shape)
print()


# article
print('##### preprocessing article fetaures... #####')
df_A_sub = df_A.drop(columns=['url', 'title', 'content'], axis=1)
df_A_sub['title_emb'] = df_A_sub.apply(lambda x: torch.tensor(x['title_emb'], dtype=torch.float32), axis=1)
df_A_sub['content_emb'] = df_A_sub.apply(lambda x: torch.tensor(x['content_emb'], dtype=torch.float32), axis=1)
#df_A_sub.to_csv('../data/features/AX.csv', index=False)
AX = encoding_attributes(df_A_sub)
torch.save(AX, '../data/features/AX_tensor.pt')
print('hashtag:', AX.shape)
print()


# image
print('##### preprocessing image fetaures... #####')
df_I_sub = df_I.drop(columns=['url', 'pixels'], axis=1)
df_I_sub = scale_numeric(df_I_sub, 'width')
df_I_sub = scale_numeric(df_I_sub, 'height')
df_I_sub['pixels_emb'] = df_I_sub.apply(lambda x: torch.tensor(x['pixels_emb'], dtype=torch.float32), axis=1)
#df_I_sub.to_csv('../data/features/IX.csv', index=False)
IX = encoding_attributes(df_I_sub)
torch.save(IX, '../data/features/IX_tensor.pt')
print('hashtag:', IX.shape)
print()

##### preprocessing claim fetaures... #####


/tmp/ipykernel_2040607/489650501.py:35: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/tmp/ipykernel_2040607/489650501.py:35: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):
/tmp/ipykernel_2040607/489650501.py:35: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):


##### Processing column  cluster  #####
##### Processing column  embedding_parsed  #####
##### Processing column  reviewers_encoded  #####
##### Processing column  date_timestamp_scaled  #####
##### Processing column  language_ohe  #####
##### Processing column  keywords_encoded  #####
##### Processing column  cluster_keywords_encoded  #####
claim: torch.Size([2168, 925])
tensor(25.)
##### preprocessing claim labels... #####
claim labels: torch.Size([2168])
##### preprocessing tweet fetaures... #####


/tmp/ipykernel_2040607/489650501.py:35: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):


##### Processing column  lang_emb  #####
##### Processing column  text_emb_parsed  #####
##### Processing column  created_at_timestamp_scaled  #####
##### Processing column  source_encoded  #####
##### Processing column  num_retweets_scaled  #####
##### Processing column  num_replies_scaled  #####
##### Processing column  num_quote_tweets_scaled  #####
tweet: torch.Size([4340, 841])

##### preprocessing reply fetaures... #####
##### Processing column  lang_emb  #####
##### Processing column  text_emb_parsed  #####
##### Processing column  created_at_timestamp_scaled  #####
##### Processing column  num_retweets_scaled  #####
##### Processing column  num_replies_scaled  #####
##### Processing column  num_quote_tweets_scaled  #####
reply: torch.Size([195459, 827])

##### preprocessing user fetaures... #####


/tmp/ipykernel_2040607/489650501.py:35: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):


##### Processing column  verified  #####
##### Processing column  protected  #####
##### Processing column  description_emb  #####
##### Processing column  created_at_timestamp_scaled  #####
##### Processing column  num_followers_scaled  #####
##### Processing column  num_followees_scaled  #####
##### Processing column  num_tweets_scaled  #####
##### Processing column  num_listed_scaled  #####
##### Processing column  location_encoded  #####
user: torch.Size([153168, 803])

##### preprocessing hashtag fetaures... #####


/tmp/ipykernel_2040607/489650501.py:35: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df[col]):


##### Processing column  tag_encoded  #####
hashtag: torch.Size([28091, 28])

##### preprocessing article fetaures... #####
##### Processing column  title_emb  #####
##### Processing column  content_emb  #####
hashtag: torch.Size([1457, 1536])

##### preprocessing image fetaures... #####
##### Processing column  pixels_emb  #####
##### Processing column  width_scaled  #####
##### Processing column  height_scaled  #####
hashtag: torch.Size([1020, 770])



In [30]:
# Load node features

CX = torch.load('../data/features/CX_tensor.pt')
CY = torch.load('../data/features/CY_tensor.pt') 
TX = torch.load('../data/features/TX_tensor.pt') 
RX = torch.load('../data/features/RX_tensor.pt')
UX = torch.load('../data/features/UX_tensor.pt')  
HX = torch.load('../data/features/HX_tensor_v2.pt') #+ clusters
AX = torch.load('../data/features/AX_tensor.pt') 
IX = torch.load('../data/features/IX_tensor.pt') 

In [31]:
print('claim:', CX.shape)
print('tweet:', TX.shape)
print('reply:', RX.shape)
print('user:', UX.shape)
print('hashtag:', HX.shape)
print('article:', AX.shape)
print('image:', IX.shape)

claim: torch.Size([2168, 925])
tweet: torch.Size([4340, 841])
reply: torch.Size([195459, 827])
user: torch.Size([153168, 803])
hashtag: torch.Size([28091, 256])
article: torch.Size([1457, 1536])
image: torch.Size([1020, 770])


In [34]:
HX

tensor([[ 2.0186e-02, -1.4105e-01,  5.3364e-02,  ..., -3.9255e-01,
         -1.6924e-01, -1.0056e+00],
        [-1.8783e-01,  1.3944e-01,  6.1717e-02,  ...,  4.5621e-01,
          8.5083e-01,  5.0169e-02],
        [-1.5533e-01, -1.0143e-03,  1.5942e-01,  ...,  1.1599e+00,
          5.2449e-01,  8.0710e-01],
        ...,
        [-7.6728e-03,  5.7329e-02,  5.6438e-02,  ...,  7.5699e-01,
          1.1308e+00, -2.8321e+00],
        [ 1.9573e-01,  1.3342e-02,  9.4734e-02,  ...,  1.4169e+00,
         -1.7235e+00, -2.2064e-02],
        [ 1.1169e-01, -1.3187e-01, -8.7259e-02,  ...,  7.7578e-01,
         -2.2761e+00,  4.6121e-01]], requires_grad=True)

In [15]:
# improve hashtag features

from sentence_transformers import SentenceTransformer
import hdbscan #from paper "Accelerated Hierarchical Density Based Clustering"


def clustering(df, col):
    threshold = 0.75
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, cluster_selection_epsilon=1-threshold)
    labels = clusterer.fit_predict(df[col].tolist())
    df['cluster_label'] = labels
    return df

def outliers_remapping(df, col):
    new_values = []
    new_value_counter = df[col].max()+1
    for value in df[col]:
        if value == -1:
            new_values.append(new_value_counter)
            new_value_counter += 1
        else:
            new_values.append(value)

    df[col+'_no_outliers'] = new_values
    return df




df_H = read_pickle_from_zip_folder('hashtag.pickle')
df_H = encoding_short_text(df_H, 'tag', target_dim=128)
df_H = clustering(df_H, 'tag_encoded')
print('All hashtags:', df_H.shape[0])
num_clusters = df_H['cluster_label'].unique().shape[0]-1
num_outliers = df_H[df_H['cluster_label']== -1].shape[0]
print('Number of clusters:', num_clusters) #3062 (con 0.75 3060)
print('Number of outliers:', num_outliers) #14365 (stesso value 0.75 e 0.85)
HX = encoding_attributes(df_H[['tag_encoded']])
torch.save(HX, '../data/features/HX_tensor.pt')
df_H = outliers_remapping(df_H, 'cluster_label')
all_ids = torch.tensor(df_H['cluster_label_no_outliers'].values)

#OHE
#df_H = one_hot_encoding(df_H, 'cluster_label') #cluster_label_no_outliers
#HX = encoding_attributes(df_H)

#EMBEDDING LAYER
embedding_dim = 128  
embedding_layer = nn.Embedding(num_clusters + num_outliers, embedding_dim)
hashtag_embeddings = embedding_layer(all_ids)

HX_augmented = torch.cat((HX, hashtag_embeddings), dim=1).detach()

torch.save(HX_augmented, '../data/features/HX_tensor_v2.pt')
print('hashtag:', HX_augmented.shape)

All hashtags: 28091
Number of clusters: 2848
Number of outliers: 15400
##### Processing column  tag_encoded  #####
hashtag: torch.Size([28091, 256])


In [10]:
# Improve image features

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import pandas as pd

# Load pre-trained ResNet model
model = models.resnet18(pretrained=True)
model = nn.Sequential(*list(model.children())[:-1])  # Remove the final classification layer
model.eval()

# Transformations for the images
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_embedding(image):
    image = preprocess(image).unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        embedding = model(image)
    return embedding.squeeze().numpy()

# the 'pixels' column contains image arrays
df_I = read_pickle_from_zip_folder('image.pickle')
df_I['pixels_encoded'] = df_I['pixels'].apply(lambda x: extract_embedding(np.array(x)))

df_I_sub = df_I.drop(columns=['url', 'pixels', 'pixels_emb'], axis=1)
df_I_sub = scale_numeric(df_I_sub, 'width')
df_I_sub = scale_numeric(df_I_sub, 'height')

IX = encoding_attributes(df_I_sub)
torch.save(IX, '../data/features/IX_tensor_v2.pt')
print('hashtag:', IX.shape)
print()

/home/martirano/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/martirano/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


##### Processing column  pixels_encoded  #####
##### Processing column  width_scaled  #####
##### Processing column  height_scaled  #####
hashtag: torch.Size([1020, 514])



In [9]:
# Improve claim features (by Paolo)

df_C = read_pickle_from_zip_folder('claim.pickle')
df_C_sub_tmp = df_C.drop(columns=['label', 'embedding', 'reviewer_emb', 'train_mask', 'val_mask', 'test_mask'], axis=1) 
#df_C_P = pd.read_parquet('/mnt/nas/pisani/dataset_mumin/claim_LP_split_60-15-25_with_embeddings.parquet')
df_C_P = pd.read_parquet('/mnt/nas/pisani/dataset_mumin/claim_LP_split_75-0-25_with_embeddings.parquet')


num_run = 5
for i in range(num_run):
    print('##### Processing run ', i)
    #df_C_sub = df_C.loc[:, ['embedding_LP_'+str(i), 'reviewers', 'date', 'language', 'keywords', 'cluster_keywords']]
    df_C_P_sub = df_C_P.loc[:, ['embedding_LP_'+str(i)]]
    df_C_P_sub_ok = df_C_P_sub.reset_index()
    df_C_sub = pd.concat([df_C_P_sub_ok, df_C_sub_tmp], axis=1)
    
    #print( df_C_P_sub_ok.index.max())
    #df_C_sub = pd.merge(df_C_sub_tmp, df_C_P_sub_ok, on='embedding_tuple')
    #df_C_sub.drop(columns=['embedding_tuple'], axis=1, inplace=True)
    print(df_C_sub.shape)
    df_C_sub = parsing_embedding_float(df_C_sub, 'embedding_LP_'+str(i))
    df_C_sub = encoding_short_text(df_C_sub, 'reviewers') #114 distinct values
    df_C_sub = convert_datetime_to_timestamp(df_C_sub, 'date')
    df_C_sub = one_hot_encoding(df_C_sub,'language') #35 distinct values
    df_C_sub = encoding_short_text(df_C_sub, 'keywords')
    df_C_sub = encoding_short_text(df_C_sub, 'cluster_keywords', target_dim=64) #26 distinct values, but multiple strings each

    CX = encoding_attributes(df_C_sub)
    #torch.save(CX, '../data/features/CX_tensor_v2_'+str(i)+'.pt')
    torch.save(CX, '../data/features/CX_tensor_v2_'+str(i)+'_no_val.pt')
    print('claim:', CX.shape)

##### Processing run  0
(2168, 8)
##### Processing column  index  #####
##### Processing column  cluster  #####
##### Processing column  embedding_LP_0_parsed  #####
##### Processing column  reviewers_encoded  #####
##### Processing column  date_timestamp_scaled  #####
##### Processing column  language_ohe  #####
##### Processing column  keywords_encoded  #####
##### Processing column  cluster_keywords_encoded  #####
claim: torch.Size([2168, 286])
##### Processing run  1
(2168, 8)
##### Processing column  index  #####
##### Processing column  cluster  #####
##### Processing column  embedding_LP_1_parsed  #####
##### Processing column  reviewers_encoded  #####
##### Processing column  date_timestamp_scaled  #####
##### Processing column  language_ohe  #####
##### Processing column  keywords_encoded  #####
##### Processing column  cluster_keywords_encoded  #####
claim: torch.Size([2168, 286])
##### Processing run  2
(2168, 8)


KeyboardInterrupt: 

In [17]:
# Load edgelists

# tweet_discusses_claim
TC_tensor = torch.load('../data/edgelists/TC_tensor.pt')
CT_tensor = torch.load('../data/edgelists/CT_tensor.pt')

# reply_reply_to_tweet
TR_r_tensor = torch.load('../data/edgelists/TR_r_tensor.pt')
RT_r_tensor = torch.load('../data/edgelists/RT_r_tensor.pt')

# reply_quote_of_tweet
TR_q_tensor = torch.load('../data/edgelists/TR_q_tensor.pt')
RT_q_tensor = torch.load('../data/edgelists/RT_q_tensor.pt')

# tweet_has_hashtag_hashtag
TH_tensor = torch.load('../data/edgelists/TH_tensor.pt')
HT_tensor = torch.load('../data/edgelists/HT_tensor.pt')

# tweet_has_article_article
TA_tensor = torch.load('../data/edgelists/TA_tensor.pt')
AT_tensor = torch.load('../data/edgelists/AT_tensor.pt')

# tweet_has_image_image
TI_tensor = torch.load('../data/edgelists/TI_tensor.pt')
IT_tensor = torch.load('../data/edgelists/IT_tensor.pt')

# tweet_mentions_user
TU_m_tensor = torch.load('../data/edgelists/TU_m_tensor.pt')
UT_m_tensor = torch.load('../data/edgelists/UT_m_tensor.pt')

# user_posted_tweet
UT_tensor = torch.load('../data/edgelists/UT_tensor.pt')
TU_tensor = torch.load('../data/edgelists/TU_tensor.pt')

# user_posted_reply
UR_tensor = torch.load('../data/edgelists/UR_tensor.pt')
RU_tensor = torch.load('../data/edgelists/RU_tensor.pt')

# user_retweeted_tweet
UT_r_tensor = torch.load('../data/edgelists/UT_r_tensor.pt')
TU_r_tensor = torch.load('../data/edgelists/TU_r_tensor.pt')

# user_follows_user
UU_f_tensor = torch.load('../data/edgelists/UU_f_tensor.pt')
UU_f_rev_tensor = torch.load('../data/edgelists/UU_f_rev_tensor.pt')

# user_mentions_user
UU_m_tensor = torch.load('../data/edgelists/UU_m_tensor.pt')
UU_m_rev_tensor = torch.load('../data/edgelists/UU_m_rev_tensor.pt')

# user_has_hashtag_hashtag
UH_tensor = torch.load('../data/edgelists/UH_tensor.pt')
HU_tensor = torch.load('../data/edgelists/HU_tensor.pt')

In [21]:
# Initialize HeteroData object
data2 = HeteroData()

# NODES

data2['claim'].x = CX
data2['claim'].y = CY
data2['tweet'].x = TX
data2['reply'].x = RX
data2['user'].x = UX
data2['hashtag'].x = HX
data2['article'].x = AX
data2['image'].x = IX

# EDGES

data2['tweet', 'discusses', 'claim'].edge_index = TC_tensor
data2['claim', 'is_discussed_by', 'tweet'].edge_index = CT_tensor

data2['reply', 'reply_to', 'tweet'].edge_index = TR_r_tensor
data2['tweet', 'is_replied_by', 'reply'].edge_index = RT_r_tensor

data2['reply', 'quote_of', 'tweet'].edge_index = TR_q_tensor
data2['tweet', 'is_quoted_by', 'reply'].edge_index = RT_q_tensor

data2['tweet', 'has_hashtag', 'hashtag'].edge_index = TH_tensor
data2['hashtag', 'is_hashtag_of', 'tweet'].edge_index = HT_tensor

data2['tweet', 'has_article', 'article'].edge_index = TA_tensor
data2['article', 'is_article_of', 'tweet'].edge_index = AT_tensor

data2['tweet', 'has_image', 'image'].edge_index = TI_tensor
data2['image', 'is_image_of', 'tweet'].edge_index = IT_tensor

data2['tweet', 'mentions', 'user'].edge_index = TU_m_tensor
data2['user', 'is_mentioned_by', 'tweet'].edge_index = UT_m_tensor

data2['user', 'posted', 'tweet'].edge_index = UT_tensor
data2['tweet', 'is_posted_by', 'user'].edge_index = TU_tensor

data2['user', 'posted', 'reply'].edge_index = UR_tensor
data2['reply', 'is_posted_by', 'user'].edge_index = RU_tensor

data2['user', 'retweeted', 'tweet'].edge_index = UT_r_tensor
data2['tweet', 'is_retweeted_by', 'user'].edge_index = TU_r_tensor

data2['user', 'follows', 'user'].edge_index = UU_f_tensor
data2['user', 'is_followed_by', 'user'].edge_index = UU_f_rev_tensor

data2['user', 'mentions', 'user'].edge_index = UU_m_tensor
data2['user', 'is_mentioned_by', 'user'].edge_index = UU_m_rev_tensor

data2['user', 'has_hashtag', 'hashtag'].edge_index = UH_tensor
data2['hashtag', 'is_hashtag_of', 'user'].edge_index = HU_tensor

In [50]:
data2

HeteroData(
  claim={
    x=[2168, 1681],
    y=[2168],
  },
  tweet={ x=[4340, 813] },
  reply={ x=[195459, 827] },
  user={ x=[153168, 1159] },
  hashtag={ x=[28091, 384] },
  article={ x=[1457, 1536] },
  image={ x=[1020, 770] },
  (tweet, discusses, claim)={ edge_index=[2, 5081] },
  (claim, is_discussed_by, tweet)={ edge_index=[2, 5081] },
  (reply, reply_to, tweet)={ edge_index=[2, 90196] },
  (tweet, is_replied_by, reply)={ edge_index=[2, 90196] },
  (reply, quote_of, tweet)={ edge_index=[2, 101216] },
  (tweet, is_quoted_by, reply)={ edge_index=[2, 101216] },
  (tweet, has_hashtag, hashtag)={ edge_index=[2, 2289] },
  (hashtag, is_hashtag_of, tweet)={ edge_index=[2, 2289] },
  (tweet, has_article, article)={ edge_index=[2, 1898] },
  (article, is_article_of, tweet)={ edge_index=[2, 1898] },
  (tweet, has_image, image)={ edge_index=[2, 1028] },
  (image, is_image_of, tweet)={ edge_index=[2, 1028] },
  (tweet, mentions, user)={ edge_index=[2, 1119] },
  (user, is_mentioned_by, tw

Data_4 --> Heterodata object with nodes/edges + meta-paths + node attributes

In [23]:
from torch_geometric.transforms import AddMetaPaths

#Add metapaths # CTUTC, CTHTC

metapaths = [[('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_posted_by', 'user'), 
              ('user', 'posted', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTUTC
             [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'has_hashtag', 'hashtag'), 
              ('hashtag', 'is_hashtag_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')],
              [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_replied_by', 'reply'), 
              ('reply', 'reply_to', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTRTC_r
              [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_quoted_by', 'reply'), 
              ('reply', 'quote_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')]] #CTRTC_q


data2 = AddMetaPaths(metapaths, weighted=True)(data2)

In [24]:
data2

HeteroData(
  metapath_dict={
    (claim, metapath_0, claim)=[4],
    (claim, metapath_1, claim)=[4],
    (claim, metapath_2, claim)=[4],
    (claim, metapath_3, claim)=[4],
  },
  claim={
    x=[2168, 925],
    y=[2168],
  },
  tweet={ x=[4340, 841] },
  reply={ x=[195459, 827] },
  user={ x=[153168, 803] },
  hashtag={ x=[28091, 256] },
  article={ x=[1457, 1536] },
  image={ x=[1020, 770] },
  (tweet, discusses, claim)={ edge_index=[2, 5081] },
  (claim, is_discussed_by, tweet)={ edge_index=[2, 5081] },
  (reply, reply_to, tweet)={ edge_index=[2, 90196] },
  (tweet, is_replied_by, reply)={ edge_index=[2, 90196] },
  (reply, quote_of, tweet)={ edge_index=[2, 101216] },
  (tweet, is_quoted_by, reply)={ edge_index=[2, 101216] },
  (tweet, has_hashtag, hashtag)={ edge_index=[2, 2289] },
  (hashtag, is_hashtag_of, tweet)={ edge_index=[2, 2289] },
  (tweet, has_article, article)={ edge_index=[2, 1898] },
  (article, is_article_of, tweet)={ edge_index=[2, 1898] },
  (tweet, has_image, imag

Data_4 --> Heterodata object with nodes/edges + meta-paths + node attributes + PCA

In [11]:
# Load node features
'''
CX = torch.load('../data/features/CX_tensor.pt')
CY = torch.load('../data/features/CY_tensor.pt') 
TX = torch.load('../data/features/TX_tensor.pt') 
RX = torch.load('../data/features/RX_tensor.pt')
UX = torch.load('../data/features/UX_tensor.pt')  
HX = torch.load('../data/features/HX_tensor_v2.pt') #+ clusters
AX = torch.load('../data/features/AX_tensor.pt') 
'''
IX = torch.load('../data/features/IX_tensor_v2.pt') 

In [12]:
#Optional: same dimnsion for all node types

from sklearn.decomposition import PCA

dim=256

pca_claim=PCA(n_components=dim).fit(CX)
torch.save(torch.tensor(pca_claim.transform(CX)).float(), '../data/features/CX_'+str(dim)+'_tensor.pt')
print('Features claim saved')

pca_tweet=PCA(n_components=dim).fit(TX)
torch.save(torch.tensor(pca_tweet.transform(TX)).float(), '../data/features/TX_'+str(dim)+'_tensor.pt')
print('Features tweet saved')

pca_reply=PCA(n_components=dim).fit(RX)
torch.save(torch.tensor(pca_reply.transform(RX)).float(), '../data/features/RX_'+str(dim)+'_tensor.pt')
print('Features reply saved')

pca_user=PCA(n_components=dim).fit(UX)
torch.save(torch.tensor(pca_user.transform(UX)).float(), '../data/features/UX_'+str(dim)+'_tensor.pt')
print('Features user saved')

pca_hashtag=PCA(n_components=dim).fit(HX)
torch.save(torch.tensor(pca_hashtag.transform(HX)).float(), '../data/features/HX_'+str(dim)+'_tensor_v2.pt')
print('Features hashtag saved')

pca_article=PCA(n_components=dim).fit(AX)
torch.save(torch.tensor(pca_article.transform(AX)).float(), '../data/features/AX_'+str(dim)+'_tensor.pt')
print('Features article saved')

pca_image=PCA(n_components=dim).fit(IX)
torch.save(torch.tensor(pca_image.transform(IX)).float(), '../data/features/IX_'+str(dim)+'_tensor.pt')
print('Features image saved')



Features image saved


In [10]:
import torch
from sklearn.decomposition import PCA

dim=256
num_run = 5

for run in range(num_run):
    print("processing run ", run+1)
    #CX = torch.load('../data/features/CX_tensor_v2_'+str(run)+'.pt')
    CX = torch.load('../data/features/CX_tensor_v2_'+str(run)+'_no_val.pt')
    pca_claim=PCA(n_components=dim).fit(CX)
    torch.save(torch.tensor(pca_claim.transform(CX)).float(), '../data/features/CX_'+str(dim)+'_tensor_v2_'+str(run)+'_no_val.pt')
print('Features claim saved')

processing run  1
processing run  2
processing run  3


KeyboardInterrupt: 

In [ ]:
# Initialize HeteroData object
data = HeteroData()

# NODES

data['claim'].x = torch.tensor(pca_claim.transform(CX)).float() #CX
data['claim'].y = CY
data['tweet'].x = torch.tensor(pca_tweet.transform(TX)).float() #TX
data['reply'].x = torch.tensor(pca_reply.transform(RX)).float() #RX
data['user'].x = torch.tensor(pca_user.transform(UX)).float() #UX
data['hashtag'].x = torch.tensor(pca_hashtag.transform(HX)).float() #HX
data['article'].x = torch.tensor(pca_article.transform(AX)).float() #AX
data['image'].x = torch.tensor(pca_image.transform(IX)).float() #IX

# EDGES

data['tweet', 'discusses', 'claim'].edge_index = TC_tensor
data['claim', 'is_discussed_by', 'tweet'].edge_index = CT_tensor

data['reply', 'reply_to', 'tweet'].edge_index = TR_r_tensor
data['tweet', 'is_replied_by', 'reply'].edge_index = RT_r_tensor

data['reply', 'quote_of', 'tweet'].edge_index = TR_q_tensor
data['tweet', 'is_quoted_by', 'reply'].edge_index = RT_q_tensor

data['tweet', 'has_hashtag', 'hashtag'].edge_index = TH_tensor
data['hashtag', 'is_hashtag_of', 'tweet'].edge_index = HT_tensor

data['tweet', 'has_article', 'article'].edge_index = TA_tensor
data['article', 'is_article_of', 'tweet'].edge_index = AT_tensor

data['tweet', 'has_image', 'image'].edge_index = TI_tensor
data['image', 'is_image_of', 'tweet'].edge_index = IT_tensor

data['tweet', 'mentions', 'user'].edge_index = TU_m_tensor
data['user', 'is_mentioned_by', 'tweet'].edge_index = UT_m_tensor

data['user', 'posted', 'tweet'].edge_index = UT_tensor
data['tweet', 'is_posted_by', 'user'].edge_index = TU_tensor

data['user', 'posted', 'reply'].edge_index = UR_tensor
data['reply', 'is_posted_by', 'user'].edge_index = RU_tensor

data['user', 'retweeted', 'tweet'].edge_index = UT_r_tensor
data['tweet', 'is_retweeted_by', 'user'].edge_index = TU_r_tensor

data['user', 'follows', 'user'].edge_index = UU_f_tensor
data['user', 'is_followed_by', 'user'].edge_index = UU_f_rev_tensor

data['user', 'mentions', 'user'].edge_index = UU_m_tensor
data['user', 'is_mentioned_by', 'user'].edge_index = UU_m_rev_tensor

data['user', 'has_hashtag', 'hashtag'].edge_index = UH_tensor
data['hashtag', 'is_hashtag_of', 'user'].edge_index = HU_tensor

In [ ]:
## Train - Val - Test split: MuMiN mask

data['claim'].train_mask = torch.tensor(df_C['train_mask'].values, dtype=torch.bool)
data['claim'].val_mask = torch.tensor(df_C['val_mask'].values, dtype=torch.bool)
data['claim'].test_mask = torch.tensor(df_C['test_mask'].values, dtype=torch.bool)

'''
## Train - Val - Test split: random

import torch_geometric.transforms as T

#transform = T.compose([T.NormalizeFeatures(), T.RandomNodeSplit()])
transform = T.RandomNodeSplit()
data2 = transform(data2)
'''

In [ ]:
from torch_geometric.transforms import AddMetaPaths

#Add metapaths # CTUTC, CTHTC

metapaths = [[('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_posted_by', 'user'), 
              ('user', 'posted', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTUTC
             [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'has_hashtag', 'hashtag'), 
              ('hashtag', 'is_hashtag_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')]] #CTHTC


data = AddMetaPaths(metapaths, weighted=True)(data)

In [ ]:
data